In [1]:
# -*- coding: utf-8 -*-
# --- Block 1: Setup and Installations ---

!pip install --upgrade transformers datasets evaluate wandb accelerate scikit-learn farasapy arabert

from google.colab import drive
import wandb
import torch
import pandas as pd
import numpy as np
import re
import torch.nn.functional as F
from tqdm import tqdm
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from arabert.preprocess import ArabertPreprocessor
from torch import nn
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ربط جوجل درايف
drive.mount('/content/drive')

# تسجيل الدخول إلى wandb
try:
    from google.colab import userdata
    wandb_api_key = userdata.get('wandp')
    if wandb_api_key:
        wandb.login(key=wandb_api_key)
        print("✅ Logged into wandb.")
    else:
        raise ValueError("No API key found in userdata.")
except Exception as e:
    from getpass import getpass
    key = getpass("Enter your wandb API key: ")
    if key:
        wandb.login(key=key)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohealdeenalkoor (mohealdeenalkoor-aiu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged into wandb.


In [2]:
# --- Block 2: Data Loading & Preprocessing ---

data_path = '/content/drive/MyDrive/coddmbined_data.csv'
df = pd.read_csv(data_path).dropna(subset=['title', 'keywords', 'abstract', 'type'])

# فلترة النصوص الإنجليزية
def is_mostly_english(text, threshold=0.55):
    if not isinstance(text, str) or not text.strip():
        return False
    words = text.split()
    total_words = len(words)
    if total_words == 0:
        return False
    english_words_count = sum(1 for w in words if re.search(r'[a-zA-Z]', w))
    return (english_words_count / total_words) > threshold

initial_count = len(df)
df = df[~df['abstract'].apply(is_mostly_english)].copy()
print(f"🗑️ تم حذف {initial_count - len(df)} مقال بسبب سيطرة اللغة الإنجليزية.")

# تعديل الفئات
classes_to_unspecify = ['المؤتمرات العلمية', 'التخطيط المستدام']
df['type'] = df['type'].replace(classes_to_unspecify, 'غير محدد')
df['type'] = df['type'].replace('بحوث الزلازل والكوارث', 'العلوم الهندسية')

unique_labels = sorted(df['type'].unique().tolist())
if 'غير محدد' not in unique_labels:
    unique_labels.append('غير محدد')

label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(id2label)
df['label'] = df['type'].map(label2id).astype(int)

# تقسيم البيانات
train_df, test_df = train_test_split(df, test_size=0.10, random_state=42, stratify=df['label'])
print(f"📊 حجم بيانات التدريب: {len(train_df)} | حجم بيانات الاختبار: {len(test_df)}")

# حفظ بيانات الاختبار
saved_test_path = '/content/drive/MyDrive/arabic_classification/extracted_test_set.csv'
test_df.to_csv(saved_test_path, index=False, encoding='utf-8')
print(f"💾 تم حفظ ملف الاختبار بنجاح في: {saved_test_path}")

# ==========================================
# ⚠️ إضافة هامة: معالجة النصوص باستخدام ArabertPreprocessor
# ==========================================
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_text_columns(dataframe):
    dataframe['title'] = dataframe['title'].apply(lambda x: arabert_prep.preprocess(str(x)))
    dataframe['keywords'] = dataframe['keywords'].apply(lambda x: arabert_prep.preprocess(str(x)))
    dataframe['abstract'] = dataframe['abstract'].apply(lambda x: arabert_prep.preprocess(str(x)))
    return dataframe

print("⏳ جاري تطبيق ArabertPreprocessor على النصوص (قد يستغرق بعض الوقت)...")
train_df = preprocess_text_columns(train_df)
test_df = preprocess_text_columns(test_df)

def combine_features(dataframe):
    sep = tokenizer.sep_token
    dataframe['full_text'] = dataframe['title'] + f" {sep} " + dataframe['keywords'] + f" {sep} " + dataframe['abstract']
    return dataframe

train_df = combine_features(train_df)
test_df = combine_features(test_df)

🗑️ تم حذف 202 مقال بسبب سيطرة اللغة الإنجليزية.
📊 حجم بيانات التدريب: 4267 | حجم بيانات الاختبار: 475
💾 تم حفظ ملف الاختبار بنجاح في: /content/drive/MyDrive/arabic_classification/extracted_test_set.csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

⏳ جاري تطبيق ArabertPreprocessor على النصوص (قد يستغرق بعض الوقت)...


In [3]:
# --- Block 3: Tokenization & Datasets ---

raw_ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['full_text', 'label']]),
    'test':  Dataset.from_pandas(test_df[['full_text', 'label']])
})

def tokenize_function(examples):
    return tokenizer(examples['full_text'], padding="max_length", truncation=True, max_length=512)

tokenized_ds = raw_ds.map(tokenize_function, batched=True)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

Map:   0%|          | 0/4267 [00:00<?, ? examples/s]

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

In [4]:
# --- Block 4: Model Training ---

wandb.init(project="arabert-arabic-classification", name="arabert_fixed_pipeline")

# حساب الأوزان
labels_array = train_df['label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_array), y=labels_array)
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cuda' if torch.cuda.is_available() else 'cpu')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/arabert_clean',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,             # تم تعديله ليكون مناسباً لـ 5 epochs
    per_device_train_batch_size=16, # تقليل الباتش لمنع OOM مع زيادة التدريب
    per_device_eval_batch_size=16,
    num_train_epochs=8,             # ⬅️ التعديل الأهم: تم رفعه لـ 5 دورات
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,
    report_to="wandb",
    run_name="arabert_clean_fixed"
)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 بدء تدريب الموديل...")
trainer.train()

save_path = "/content/drive/MyDrive/arabert_clean_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
wandb.finish()
print(f"🏁 تم الانتهاء وحفظ أفضل موديل بنجاح في: {save_path}")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 بدء تدريب الموديل...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.789265,0.778947,0.701581
2,0.926695,0.625936,0.848421,0.765887
3,0.926695,0.572082,0.871579,0.785894
4,0.335819,0.601136,0.886316,0.800081
5,0.335819,0.633293,0.888421,0.802042
6,0.166437,0.686449,0.886316,0.803143
7,0.166437,0.697873,0.884211,0.801642


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 